# Bildprojekt · Ziffernnetz als kurzer Einstieg (1. Klasse)
**DL 4 · ca. 20 Minuten.** Ein vorhandenes Ziffernnetz hat Ausgabeklassen 0–9. Testet zwei Ziffern und die schematischen Buchstaben A/B und erklärt die Grenze.

**Wichtig:** Für A/B gibt das Modell trotzdem eine Ziffer aus – es hat keine Buchstabenausgabe. Ein zweiter Buchstaben-Trainingsdurchlauf entfällt. In DL 5 trainieren wir statt dessen einen neuen Ausgabekopf mit festen MobileNet-Merkmalen.

In [ ]:
import sys
if sys.platform == "emscripten":
    import pyodide_js
    await pyodide_js.loadPackage(["numpy","scipy","scikit-learn","matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
rng=np.random.default_rng(42)
digits=load_digits()
X=digits.data.astype(float)/16
y=digits.target
# Separate train and test images; test examples never enter training.
indices=rng.permutation(len(y)); train=indices[:1400]; test=indices[1400:]
digit_model=MLPClassifier(hidden_layer_sizes=(32,),max_iter=180,random_state=12,early_stopping=True)
digit_model.fit(X[train],y[train])
print("Ziffern: Testtrefferquote",round(accuracy_score(y[test],digit_model.predict(X[test])),2))
def zeigen(bilder,titel):
    fig,axs=plt.subplots(1,len(bilder),figsize=(2*len(bilder),2))
    for ax,img,txt in zip(np.atleast_1d(axs),bilder,titel):
        ax.imshow(img.reshape(8,8),cmap="gray",vmin=0,vmax=1);ax.set_title(str(txt));ax.axis("off")
    plt.show()
zeigen(X[test[:5]],digit_model.predict(X[test[:5]]))

## DL 4 · Was passiert mit unbekannten Buchstaben?
Vermutet zuerst die Ausgabe für A und B. Führt danach die Zelle aus und haltet die vorhergesagten **Ziffern** fest. Die Zeichen sind schematische 8×8-Bilder, keine echte Handschriftdatensammlung.

In [ ]:
A_rows=["00111000","01000100","01000100","01111100","01000100","01000100","01000100","00000000"]
B_rows=["01111000","01000100","01000100","01111000","01000100","01000100","01111000","00000000"]
def raster(rows):
    return np.array([[int(c) for c in row] for row in rows],dtype=float)
A=raster(A_rows); B=raster(B_rows)
zeigen([A,B],["A","B"])
print("Ziffernnetz sagt für A und B:",digit_model.predict(np.array([A.ravel(),B.ravel()])))

**Sicherung auf der Kursseite:** Eine Ziffernvorhersage bei A/B ist keine richtige Buchstabenerkennung. Um neue Klassen zu erkennen, braucht das Modell einen dafür trainierten Ausgabekopf. In unserem eigentlichen Bildprojekt lernt nur dieser kleine Kopf.